# AI Outputs — Analysis & Visualization
### Companion notebook for the lecture *"Advanced Data Analysis & Visualization for AI Outputs"*

---

| # | Topic | What you will do |
|---|-------|------------------|
| 1 | **Performance metrics** | Measure a model beyond simple accuracy |
| 2 | **Explainability (LIME)** | Understand individual predictions using a local interpretable surrogate |
| 3 | **Dimensionality reduction** | Compare PCA, t-SNE, and UMAP side by side |
| 4 | **Distribution shift** | Detect when new data looks different from training data |

No prior programming experience is required beyond being able to run cells.
Each section opens with a plain-language explanation before any code.

---
## Setup — run this cell first

This cell loads all the Python libraries used throughout the notebook.

> **If any library is missing**, install it from a terminal with:
> ```
> pip install lime umap-learn
> ```

In [ ]:
# --- Standard scientific Python stack ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# --- Machine-learning utilities (scikit-learn) ---
from sklearn.datasets import make_classification, load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, roc_auc_score,
    precision_score, recall_score, f1_score
)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# --- UMAP (non-linear dimensionality reduction) ---
import umap                          # pip install umap-learn

# --- LIME explainability ---
import lime
import lime.lime_tabular             # pip install lime

# --- Statistical tests ---
from scipy import stats

# Consistent plot style
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False,
                     'axes.spines.right': False})
SEED = 42
np.random.seed(SEED)
print('All libraries loaded successfully.')
# --- Text classification (Section 2b) ---
from sklearn.datasets import fetch_20newsgroups
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
import lime.lime_text

---
## 1 — Performance Metrics: Beyond Accuracy

Accuracy answers "how often is the model right overall?"  
That is misleading whenever one class is rare.  
Imagine a disease affecting 1 in 100 people: a model that always predicts *no disease* achieves 99 % accuracy — yet catches zero cases.

Two more informative views:
* **Confusion matrix** — a 2×2 table counting correct and incorrect predictions for each class.
* **ROC curve** — plots the trade-off between catching true positives and generating false alarms across every possible decision threshold. The area under the curve (AUC) summarises it in one number: 1.0 is perfect, 0.5 is random.

In [ ]:
# --- Synthetic binary dataset with mild class imbalance ---
# weights=[0.7, 0.3] means class 1 appears 30 % of the time — a realistic scenario.
X, y = make_classification(
    n_samples=600, n_features=10, weights=[0.7, 0.3], random_state=SEED
)

# --- Split: 75 % training, 25 % test ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED
)

# --- Scale features ---
# Logistic regression is sensitive to scale. StandardScaler centres each feature.
scaler  = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# --- Train ---
clf     = LogisticRegression(random_state=SEED)
clf.fit(X_train_s, y_train)

# --- Predict ---
y_pred = clf.predict(X_test_s)
y_prob = clf.predict_proba(X_test_s)[:, 1]  # probability of class 1

print(f"Precision : {precision_score(y_test, y_pred):.2f}  "
      "— of all predicted positives, how many are real?")
print(f"Recall    : {recall_score(y_test, y_pred):.2f}  "
      "— of all real positives, how many did we catch?")
print(f"F1 Score  : {f1_score(y_test, y_pred):.2f}  "
      "— harmonic mean of precision and recall")
print(f"ROC-AUC   : {roc_auc_score(y_test, y_prob):.2f}  "
      "— 1.0 is perfect; 0.5 is random")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# --- Confusion matrix ---
# Diagonal cells = correct predictions. Off-diagonal = errors.
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred)).plot(
    ax=axes[0], colorbar=False
)
axes[0].set_title('Confusion Matrix\n(diagonal = correct predictions)')

# --- ROC curve ---
# The dashed line is a random classifier; our curve should be well above it.
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)
axes[1].plot(fpr, tpr, lw=2, label=f'Model  (AUC = {auc:.2f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC = 0.50)')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()

plt.tight_layout()
plt.show()

# --- Precision-Recall curve and AUPRC ---
# The PR curve is more informative than ROC when the positive class is rare.
# A random classifier's baseline here is not 0.5 but the positive class prevalence.
from sklearn.metrics import precision_recall_curve, average_precision_score

precision, recall, _ = precision_recall_curve(y_test, y_prob)
auprc    = average_precision_score(y_test, y_prob)

# Positive class prevalence = the no-skill baseline for a random classifier
prevalence = y_test.mean()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# --- Left: PR curve ---
# A perfect classifier hugs the top-right corner (precision=1, recall=1).
# The dashed horizontal line is the baseline: predicting positive at random.
axes[0].plot(recall, precision, lw=2, color='steelblue',
             label=f'Model  (AUPRC = {auprc:.2f})')
axes[0].axhline(prevalence, color='k', linestyle='--', lw=1,
                label=f'Random (prevalence = {prevalence:.2f})')
axes[0].set_xlabel('Recall  (fraction of true positives caught)')
axes[0].set_ylabel('Precision  (fraction of predicted positives correct)')
axes[0].set_title('Precision-Recall Curve')
axes[0].set_xlim([0, 1])
axes[0].set_ylim([0, 1.05])
axes[0].legend()

# --- Right: ROC vs PR side-by-side summary bar ---
# AUC = 1 is perfect; AUPRC baseline depends on class prevalence,
# so the two numbers are not directly comparable.
metrics = {
    'ROC-AUC\n(baseline 0.50)':   roc_auc_score(y_test, y_prob),
    f'AUPRC\n(baseline {prevalence:.2f})': auprc,
}
colours = ['#4878CF', '#D65F5F']
bars = axes[1].bar(metrics.keys(), metrics.values(),
                   color=colours, edgecolor='white', width=0.4)
axes[1].set_ylim(0, 1.05)
axes[1].set_ylabel('Score')
axes[1].set_title('ROC-AUC vs. AUPRC\n(higher is better for both)')

# Annotate bars with exact values
for bar, val in zip(bars, metrics.values()):
    axes[1].text(bar.get_x() + bar.get_width() / 2,
                 val + 0.02, f'{val:.2f}',
                 ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

print('When the positive class is rare, AUPRC is the more demanding metric:')
print(f'  A random classifier achieves ROC-AUC ≈ 0.50 regardless of class balance.')
print(f'  A random classifier achieves AUPRC  ≈ {prevalence:.2f} (= positive class rate).')
print('  A good model must beat both baselines by a clear margin.')

---
## 2 — Explainability: Why Did the Model Predict That?

A model can be highly accurate and still be untrustworthy if no one can
explain its individual decisions.  
**LIME** (Ribeiro et al., 2016) explains any black-box model
one prediction at a time.

The key idea:
> *To explain a single prediction, LIME creates many slightly
> modified versions of the input, runs them all through the
> black-box model, and fits a simple linear model to the results.
> The weights of that simple model reveal which features drove
> the prediction.*

Positive weights push the prediction toward class 1;  
negative weights push it toward class 0.

We use a **Random Forest** here — a genuinely non-linear model that
cannot be inspected directly, which makes LIME's role more meaningful
than on a logistic regression.

In [ ]:
# --- Train a Random Forest on the same data ---
# Random Forests combine many decision trees; their predictions are opaque.
feature_names = [f'Feature {i+1}' for i in range(X_train.shape[1])]

rf = RandomForestClassifier(n_estimators=100, random_state=SEED)
rf.fit(X_train_s, y_train)

print(f'Random Forest test accuracy: {rf.score(X_test_s, y_test):.2f}')
print()
print('Now we will use LIME to explain individual predictions from this model.')

In [ ]:
# --- Build the LIME explainer ---
# The explainer needs to know about the training data distribution
# so it can generate realistic perturbed versions of any input.
explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data  = X_train_s,
    feature_names  = feature_names,
    class_names    = ['Class 0', 'Class 1'],
    mode           = 'classification',
    random_state   = SEED
)

# --- Choose a test example to explain ---
# Change this index to inspect a different prediction.
EXAMPLE_IDX = 5

instance = X_test_s[EXAMPLE_IDX]
true_label      = y_test[EXAMPLE_IDX]
predicted_label = rf.predict([instance])[0]
predicted_prob  = rf.predict_proba([instance])[0, 1]

print(f'Example index   : {EXAMPLE_IDX}')
print(f'True label      : {true_label}')
print(f'Predicted label : {predicted_label}')
print(f'P(Class 1)      : {predicted_prob:.2f}')
print()

# --- Generate the LIME explanation ---
# num_features: how many features to include in the local linear model.
# num_samples:  how many perturbed versions of the input to generate.
explanation = explainer.explain_instance(
    data_row    = instance,
    predict_fn  = rf.predict_proba,
    num_features= 10,
    num_samples = 1000
)

print('Explanation generated. Run the next cell to see the plot.')

In [ ]:
# --- Plot 1: LIME bar chart for a single prediction ---
# Each bar shows one feature's contribution to this prediction.
# Orange bars push toward Class 1; blue bars push toward Class 0.
fig, ax = plt.subplots(figsize=(8, 4))

weights = explanation.as_list()                  # list of (condition_string, weight)
labels  = [w[0] for w in weights]
values  = [w[1] for w in weights]
colours = ['#D65F5F' if v > 0 else '#4878CF' for v in values]

# Sort by absolute weight so the most important feature is at the top
order  = np.argsort(np.abs(values))
labels = [labels[i] for i in order]
values = [values[i] for i in order]
colours= [colours[i] for i in order]

ax.barh(labels, values, color=colours, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('LIME weight  (positive = pushes toward Class 1)')
ax.set_title(
    f'LIME Explanation — Example {EXAMPLE_IDX}\n'
    f'True: {true_label}  |  Predicted: {predicted_label}  |  P(Class 1) = {predicted_prob:.2f}'
)
plt.tight_layout()
plt.show()
print('Red bars pushed the model toward predicting Class 1.')
print('Blue bars pushed it toward Class 0.')

In [ ]:
# --- Plot 2: LIME explanations for several predictions side by side ---
# This shows how LIME gives a different explanation for each individual input —
# the same feature can matter a lot for one example and very little for another.

EXAMPLES = [0, 1, 5, 10, 20, 35]   # which test examples to explain
TOP_N    = 6                         # show only the top-N features per example

fig, axes = plt.subplots(2, 3, figsize=(14, 7))

for ax, idx in zip(axes.flat, EXAMPLES):
    inst   = X_test_s[idx]
    t_lbl  = y_test[idx]
    p_lbl  = rf.predict([inst])[0]
    p_prob = rf.predict_proba([inst])[0, 1]

    exp = explainer.explain_instance(
        data_row   = inst,
        predict_fn = rf.predict_proba,
        num_features= TOP_N,
        num_samples = 500
    )

    w      = exp.as_list()
    lbls   = [x[0].split('<=')[0].split('>')[0].strip() for x in w]  # shorten condition
    vals   = [x[1] for x in w]
    cols   = ['#D65F5F' if v > 0 else '#4878CF' for v in vals]

    # Sort by absolute weight
    order  = np.argsort(np.abs(vals))
    ax.barh([lbls[i] for i in order], [vals[i] for i in order],
            color=[cols[i] for i in order], edgecolor='white')
    ax.axvline(0, color='black', linewidth=0.6)
    ax.set_title(
        f'Ex {idx} — True:{t_lbl} / Pred:{p_lbl}\nP(C1)={p_prob:.2f}',
        fontsize=9
    )
    ax.tick_params(axis='y', labelsize=7)
    ax.tick_params(axis='x', labelsize=7)

plt.suptitle(
    'LIME explanations for 6 different predictions\n'
    'The same feature plays a different role in each case',
    fontsize=11, y=1.01
)
plt.tight_layout()
plt.show()

---
### 2b — LIME on Text Classification

`LimeTabularExplainer` operates on feature vectors. `LimeTextExplainer` applies the same local-surrogate principle directly to raw text: it perturbs a document by randomly toggling word presence, observes how the classifier’s output changes across the perturbed neighbourhood, and fits a sparse linear model over the binary word-indicator features. The resulting weights identify which words push the prediction toward each class for that specific instance.

We use the **20 Newsgroups** corpus, restricting to two topically distant categories (`sci.space` and `rec.sport.hockey`) so that the learned vocabulary is interpretable. The pipeline is a standard TF-IDF representation followed by logistic regression — a transparent baseline that lets us verify whether LIME’s word attributions agree with our own reading of the document.


In [ ]:
# -- Fetch a binary subset of 20 Newsgroups -----------------------------------
# Two categories chosen for maximal topical distance; this keeps the
# decision boundary sharp and the LIME word attributions easy to audit.
CATEGORIES = ['sci.space', 'rec.sport.hockey']

news_train = fetch_20newsgroups(
    subset='train', categories=CATEGORIES,
    remove=('headers', 'footers', 'quotes'),  # strip metadata that leaks the label
    random_state=SEED
)
news_test = fetch_20newsgroups(
    subset='test', categories=CATEGORIES,
    remove=('headers', 'footers', 'quotes'),
    random_state=SEED
)

CLASS_NAMES = news_train.target_names   # ['rec.sport.hockey', 'sci.space']
print(f'Training documents : {len(news_train.data)}')
print(f'Test documents     : {len(news_test.data)}')
print(f'Classes            : {CLASS_NAMES}')
print()

# -- Build and train the pipeline ---------------------------------------------
# TfidfVectorizer converts raw text to a tf-idf weighted term-document matrix.
# sublinear_tf=True applies log(1+tf) scaling, reducing the dominance of very
# frequent terms without removing them entirely.
text_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=5000,
        sublinear_tf=True,
        stop_words='english'
    )),
    ('clf', LogisticRegression(max_iter=1000, random_state=SEED))
])

text_pipeline.fit(news_train.data, news_train.target)

test_acc = text_pipeline.score(news_test.data, news_test.target)
print(f'Test accuracy : {test_acc:.3f}')
print()
print('Pipeline trained. Next cell generates a LIME explanation for one document.')


In [ ]:
# -- Select a test document to explain ----------------------------------------
# Change TEXT_IDX to inspect a different document.
TEXT_IDX = 5

doc        = news_test.data[TEXT_IDX]
true_label = news_test.target[TEXT_IDX]
pred_proba = text_pipeline.predict_proba([doc])[0]
pred_label = text_pipeline.predict([doc])[0]

print(f'Document index  : {TEXT_IDX}')
print(f'True class      : {CLASS_NAMES[true_label]}  (label {true_label})')
print(f'Predicted class : {CLASS_NAMES[pred_label]}  (label {pred_label})')
proba_str = {CLASS_NAMES[i]: f"{p:.3f}" for i, p in enumerate(pred_proba)}
print(f'Class probs     : {proba_str}')
print()
# Truncated view for reference
print('--- Document (first 400 characters) ---')
print(doc[:400].strip())
print()

# -- Build the LimeTextExplainer ----------------------------------------------
# No training data required: the explainer perturbs the text instance directly
# by hiding random subsets of words and re-querying predict_proba each time.
text_explainer = lime.lime_text.LimeTextExplainer(
    class_names=CLASS_NAMES,
    random_state=SEED
)

# explain_instance fits a local linear model over binary word-presence features.
# num_features: words retained in the surrogate model.
# num_samples:  neighbourhood size (more = more stable, but slower).
text_exp = text_explainer.explain_instance(
    text_instance=doc,
    classifier_fn=text_pipeline.predict_proba,
    num_features=12,
    num_samples=2000,
    labels=(pred_label,)   # explain the predicted class only
)

print('Explanation generated. Run the next cell to visualise the word weights.')


In [ ]:
# -- Plot: LIME word attributions for the selected document -------------------
# Each bar corresponds to one word.
# A positive weight supports the predicted class; a negative weight opposes it.

word_weights = text_exp.as_list(label=pred_label)  # list of (word, weight)

words  = [w[0] for w in word_weights]
values = [w[1] for w in word_weights]

# Sort by absolute contribution (most influential at the top)
order  = np.argsort(np.abs(values))
words  = [words[i]  for i in order]
values = [values[i] for i in order]
colors = ['#D65F5F' if v > 0 else '#4878CF' for v in values]

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(words, values, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel(
    f'LIME weight  (positive = evidence for "{CLASS_NAMES[pred_label]}")'
)
ax.set_title(
    f'LIME Text Explanation — Document {TEXT_IDX}\n'
    f'True: {CLASS_NAMES[true_label]}  |  '
    f'Predicted: {CLASS_NAMES[pred_label]}  |  '
    f'P = {pred_proba[pred_label]:.2f}'
)

# Annotate bars with exact weights
for i, (val, word) in enumerate(zip(values, words)):
    ax.text(
        val + (0.002 if val >= 0 else -0.002),
        i,
        f'{val:+.3f}',
        va='center',
        ha='left' if val >= 0 else 'right',
        fontsize=8
    )

plt.tight_layout()
plt.show()

print('Red bars: words supporting the predicted class.')
print('Blue bars: words arguing against the predicted class.')
print()
print(
    'Unlike the tabular case, the features here are directly human-readable:\n'
    'you can verify each attribution by locating the word in the document above.'
)


---
## 3 — Dimensionality Reduction: PCA vs. t-SNE vs. UMAP

High-dimensional data cannot be plotted directly. Dimensionality
reduction compresses many features into two dimensions while
preserving as much structure as possible.

We compare three methods on the **Digits dataset** (1,797 handwritten
digit images, each represented as 64 pixel values — 64 dimensions):

| Method | Type | Preserves | Speed |
|--------|------|-----------|-------|
| **PCA** | Linear | Global distances | Fast |
| **t-SNE** | Non-linear | Local clusters | Slow |
| **UMAP** | Non-linear | Both local and global | Fast |

A good 2D projection shows distinct, well-separated clusters —
one per digit class.

In [ ]:
# --- Load the Digits dataset ---
# 1,797 examples × 64 features (8×8 pixel images of digits 0–9)
digits  = load_digits()
X_dig   = digits.data           # shape: (1797, 64)
y_dig   = digits.target         # class labels: 0–9

# --- Scale before any projection ---
# All three methods are sensitive to feature scale;
# StandardScaler prevents high-variance pixels from dominating.
X_dig_s = StandardScaler().fit_transform(X_dig)

print(f'Dataset shape : {X_dig.shape}')
print(f'Classes       : {np.unique(y_dig)}')
print()
print('Running three projections — this may take 20–60 seconds...')

# --- PCA: linear, deterministic, fast ---
pca_proj = PCA(n_components=2, random_state=SEED).fit_transform(X_dig_s)
print('  PCA done.')

# --- t-SNE: non-linear, focuses on local structure ---
# perplexity ≈ expected cluster size; 30 is a common default.
tsne_proj = TSNE(
    n_components=2, perplexity=30, random_state=SEED, n_iter=1000
).fit_transform(X_dig_s)
print('  t-SNE done.')

# --- UMAP: non-linear, faster than t-SNE, preserves more global structure ---
# n_neighbors controls the balance between local and global structure.
umap_proj = umap.UMAP(
    n_components=2, n_neighbors=15, min_dist=0.1, random_state=SEED
).fit_transform(X_dig_s)
print('  UMAP done.')

In [ ]:
# --- Side-by-side comparison ---
# Each dot is one digit image; colour encodes the true class (0–9).
# Well-separated clusters of the same colour = good projection.

CMAP    = plt.get_cmap('tab10')
methods = [
    ('PCA',   pca_proj,  'Linear projection — preserves global structure but clusters overlap'),
    ('t-SNE', tsne_proj, 'Non-linear — tight local clusters, global distances not meaningful'),
    ('UMAP',  umap_proj, 'Non-linear — tight clusters with more faithful global layout'),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, proj, subtitle) in zip(axes, methods):
    for cls in range(10):
        mask = y_dig == cls
        ax.scatter(
            proj[mask, 0], proj[mask, 1],
            color=CMAP(cls), label=str(cls),
            s=8, alpha=0.7, edgecolors='none'
        )
    ax.set_title(f'{name}\n{subtitle}', fontsize=9)
    ax.set_xlabel('Component 1')
    ax.set_ylabel('Component 2')
    ax.tick_params(labelleft=False, labelbottom=False)

# Single shared legend
handles = [plt.Line2D([0],[0], marker='o', color=CMAP(i),
           linestyle='none', markersize=6, label=str(i)) for i in range(10)]
fig.legend(handles=handles, title='Digit', ncol=10,
           loc='lower center', bbox_to_anchor=(0.5, -0.06), fontsize=8)

plt.suptitle('Digits dataset (64 features → 2D) — Three projection methods',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- PCA scree plot: how many dimensions are really needed? ---
# Fit PCA keeping all components to see the full variance profile.
pca_full = PCA(random_state=SEED).fit(X_dig_s)
explained = pca_full.explained_variance_ratio_
cumulative = np.cumsum(explained)

# Find where cumulative variance passes 90 %
n_90 = np.searchsorted(cumulative, 0.90) + 1
print(f'{n_90} components explain 90 % of the variance (out of 64 total).')

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(1, len(explained)+1), explained * 100,
       color='steelblue', alpha=0.6, label='Per component')
ax.plot(range(1, len(cumulative)+1), cumulative * 100,
        'o-', color='coral', markersize=3, label='Cumulative')
ax.axhline(90, color='grey', linestyle='--', linewidth=0.8, label='90 % threshold')
ax.axvline(n_90, color='grey', linestyle=':', linewidth=0.8)
ax.set_xlabel('Principal component')
ax.set_ylabel('Variance explained (%)')
ax.set_title(f'Scree plot — {n_90} components needed for 90 % of variance')
ax.set_xlim(0, 40)      # zoom in on the informative region
ax.legend()
plt.tight_layout()
plt.show()

---
## 4 — Distribution Shift: When Production Data Looks Different

A model is trained on historical data. If the data it encounters in
production changes — different season, different user population,
equipment drift — its predictions may degrade without any visible error signal.

The **Kolmogorov-Smirnov (KS) test** compares two distributions and
returns a *p*-value. A small *p* (< 0.05) is evidence that the two
samples come from different distributions — shift detected.

The KS statistic is the largest gap between the two
cumulative distribution curves — we will mark it visually.

In [ ]:
n = 800

# --- Simulate training and production distributions ---
# Training data is centred at 0; production data has drifted to 0.8.
train_feat = np.random.normal(loc=0.0, scale=1.0, size=n)
prod_feat  = np.random.normal(loc=0.8, scale=1.0, size=n)

# --- KS test ---
# H0: both samples come from the same distribution.
# Small p-value → reject H0 → distribution shift detected.
ks_stat, p_val = stats.ks_2samp(train_feat, prod_feat)

print(f'KS statistic : {ks_stat:.3f}   (maximum gap between the two CDFs)')
print(f'p-value      : {p_val:.4f}')
print()
if p_val < 0.05:
    print('⚠  p < 0.05 — distribution shift detected.')
else:
    print('✓  p ≥ 0.05 — no significant shift at the 5 % level.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# --- Left: overlapping histograms ---
axes[0].hist(train_feat, bins=40, alpha=0.55, color='steelblue', label='Training')
axes[0].hist(prod_feat,  bins=40, alpha=0.55, color='coral',     label='Production')
axes[0].set_xlabel('Feature value')
axes[0].set_ylabel('Count')
axes[0].set_title('Feature Distribution\nTraining vs. Production')
axes[0].legend()

# --- Right: CDFs with KS gap marked ---
# The CDFs show the cumulative fraction of values up to each point.
# The KS statistic is the largest vertical distance between the two curves.
x_vals = np.sort(np.concatenate([train_feat, prod_feat]))
cdf_tr = np.array([np.mean(train_feat <= v) for v in x_vals])
cdf_pr = np.array([np.mean(prod_feat  <= v) for v in x_vals])

axes[1].plot(x_vals, cdf_tr, color='steelblue', lw=1.5, label='Training CDF')
axes[1].plot(x_vals, cdf_pr, color='coral',     lw=1.5, label='Production CDF')

# Mark the KS gap
gap_idx = np.argmax(np.abs(cdf_tr - cdf_pr))
axes[1].vlines(
    x_vals[gap_idx], cdf_tr[gap_idx], cdf_pr[gap_idx],
    color='black', lw=2, label=f'KS gap = {ks_stat:.3f}'
)
axes[1].set_xlabel('Feature value')
axes[1].set_ylabel('Cumulative fraction')
axes[1].set_title('CDFs with KS Statistic Marked')
axes[1].legend()

plt.tight_layout()
plt.show()
print('The black vertical bar shows the largest gap — that is the KS statistic.')